# Gold layer for building a data model (STAR schema)
### The transformation logic will include one SQL query, which will create a table for the gold schema in the catalog

In [0]:
# First join with ERP customer MD, so I can get their locations from the ERP system
query = """
SELECT
    ROW_NUMBER() OVER (ORDER BY ci.Customer_ID) AS row_id,
    ci.Customer_ID AS customer_id,
    ci.Customer_Key AS customer_key,
    ci.First_Name AS first_name,
    ci.Last_Name AS last_name,
    la.country,
    ci.Marital_Status AS marital_status,
    CASE
        WHEN ci.Gender <> 'n/a' THEN ci.Gender
        ELSE COALESCE(ca.gender, 'n/a')
    END AS gender,
    ca.birth_date AS birthdate,
    ci.Create_Date AS create_date
FROM workspace.silver.crm_customer ci
LEFT JOIN workspace.silver.erp_customer ca 
    ON ci.Customer_Key = ca.customer_number
LEFT JOIN workspace.silver.erp_locations la
    ON ci.Customer_Key = la.customer_number
"""

df = spark.sql(query)
df.display()

## Write the table to the gold schema of the catalog

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.gold.dim_customers")